# Baseline Models

## Objective

The objective of this notebook is to establish reliable baseline performance for the Telco Churn classification problem.

This notebook will:

- train a `DummyClassifier` to establish a minimum reference performance;
- train Logistic Regression as a simple, interpretable linear baseline;
- train a Decision Tree as a nonlinear baseline;
- generate class predictions and churn probabilities;
- create a reusable evaluation function;
- evaluate each model using accuracy, precision, recall, F1-score, balanced accuracy, ROC-AUC, and PR-AUC;
- analyze confusion matrices, with particular attention to false negatives;
- compare training and validation performance to identify possible overfitting;
- summarize the initial strengths and limitations of each baseline model.

Because the target is imbalanced, accuracy will not be used as the only evaluation criterion. Particular attention will be given to recall and F1-score for the churn class because failing to identify customers who are likely to leave may represent an important business cost.

Model comparison will be performed using only the training data and cross-validation. The test set will remain untouched until the final model and decision threshold have been selected.

In [43]:
import pandas as pd
from pathlib import Path
from sklearn.pipeline import Pipeline
import numpy as np
from IPython.display import display
import joblib

# Models
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

## Load modeling data and fitted preprocessor

The audited dataset is loaded and divided using the same configuration established in the preprocessing notebook. Because the split uses a fixed `random_state` and stratification, it reproduces the same training and test observations.

The previously fitted preprocessor is then loaded and used to transform both datasets. It is not refitted in this notebook, ensuring that the transformations learned exclusively from the training data remain unchanged.

The feature metadata is also loaded to assign meaningful column names to the transformed matrices.

In [44]:
SPLIT_DATA_DIR = Path("../data/processed/splits")

train_data = pd.read_csv(SPLIT_DATA_DIR / "train.csv")
test_data = pd.read_csv(SPLIT_DATA_DIR / "test.csv")

X_train = train_data.drop(columns="Churn")
y_train = train_data["Churn"]

X_test = test_data.drop(columns="Churn")
y_test = test_data["Churn"]

In [45]:
PROJECT_ROOT = Path("..")
DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DATA_PATH = DATA_DIR / "processed" / "column_audit.csv"
if not PROCESSED_DATA_PATH.exists():
    raise FileNotFoundError(f"Dataset not found at {PROCESSED_DATA_PATH.resolve()}")
column_audit = pd.read_csv(PROCESSED_DATA_PATH, index_col=0)
column_audit

,column,dtype,unique_count,possible_role
0,customerID,str,7043,identifier
1,gender,str,2,binary categorical
2,SeniorCitizen,int64,2,binary categorical stored as integer
3,Partner,str,2,binary categorical
4,Dependents,str,2,binary categorical
5,tenure,int64,73,numeric
6,PhoneService,str,2,binary categorical
7,MultipleLines,str,3,multiclass categorical
8,InternetService,str,3,multiclass categorical
9,OnlineSecurity,str,3,multiclass categorical


In [46]:
expected_features = column_audit.loc[~column_audit["column"].isin(["customerID", "Churn"]),"column"].copy()
n_expected_features = len(expected_features)

print(expected_features)
print("Number of expected features:", n_expected_features)

1               gender
2        SeniorCitizen
3              Partner
4           Dependents
5               tenure
6         PhoneService
7        MultipleLines
8      InternetService
9       OnlineSecurity
10        OnlineBackup
11    DeviceProtection
12         TechSupport
13         StreamingTV
14     StreamingMovies
15            Contract
16    PaperlessBilling
17       PaymentMethod
18      MonthlyCharges
19        TotalCharges
Name: column, dtype: str
Number of expected features: 19


In [47]:
MODELS_DIR = Path("../models")
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.joblib"
if not PREPROCESSOR_PATH.exists():
    raise FileNotFoundError(f"preprocessor not found at {PREPROCESSOR_PATH.resolve()}")
preprocessor = joblib.load(PREPROCESSOR_PATH)

In [48]:
assert "customerID" not in expected_features, "customerID should have been excluded"
assert "Churn" not in expected_features, "Churn should have been excluded"
assert len(expected_features) == len(column_audit["column"]) - 2, "Should be total columns minus customerID and Churn"

In [49]:
assert set(X_train.columns) == set(expected_features), (
    f"Mismatch in X_train columns.\n"
    f"Missing: {set(expected_features) - set(X_train.columns)}\n"
    f"Extra: {set(X_train.columns) - set(expected_features)}"
)

assert set(X_test.columns) == set(expected_features), (
    f"Mismatch in X_test columns.\n"
    f"Missing: {set(expected_features) - set(X_test.columns)}\n"
    f"Extra: {set(X_test.columns) - set(expected_features)}"
)

### Feature schema validation results

The expected predictor schema was validated successfully before model training.

The `customerID` identifier and `Churn` target are excluded from the predictor set, leaving 19 expected input features. Both `X_train` and `X_test` contain exactly these features, with no missing or unexpected columns.

This confirms that the modeling data is structurally consistent with the schema used to build the fitted preprocessor. The validated feature set can now be transformed safely using the previously saved preprocessing artifact.

## Baseline model pipelines

A consistent machine-learning pipeline will be created for each baseline classifier. Each pipeline combines the preprocessing stage with a model, allowing the classifiers to receive the same feature transformations and ensuring a fair comparison.

The baseline models included are:

- `DummyClassifier`, which establishes the minimum reference performance;
- `LogisticRegression`, which provides a simple linear baseline;
- `DecisionTreeClassifier`, which captures nonlinear relationships;
- `RandomForestClassifier`, which combines multiple decision trees to improve stability and generalization.

The preprocessing and model steps are kept together to reduce the risk of applying inconsistent transformations. During cross-validation, the preprocessing stage must be fitted separately within each training fold to prevent information from the validation folds from influencing the transformations.

In [50]:
display(type(X_train), X_train.shape)
display(type(y_train), y_train.shape)
display(type(preprocessor))

pandas.DataFrame

(5634, 19)

pandas.Series

(5634,)

sklearn.compose._column_transformer.ColumnTransformer

In [51]:
model = {
    "dummy": DummyClassifier(),
    "logistic_regression": LogisticRegression(),
    "decision_tree": DecisionTreeClassifier(),
    "random_forest": RandomForestClassifier()
}

In [53]:
baseline_pipeline = {
    model_name: Pipeline([
        ('preprocessing', preprocessor),
        ('model', estimator)
    ])
    for model_name, estimator in model.items()
}

### Baseline model pipelines

A separate machine-learning pipeline was created for each baseline model. Every pipeline contains two stages:

1. The fitted preprocessing configuration transforms the original customer features.
2. A classifier receives the transformed features and produces churn predictions.

Four baseline models were included for comparison:

- **Dummy Classifier**: predicts the majority class regardless of input; establishes the minimum performance any real model must exceed.
- **Logistic Regression**: a simple, interpretable linear model, commonly used as a first real baseline for binary classification.
- **Decision Tree**: captures non-linear relationships and feature interactions without requiring scaled inputs.
- **Random Forest**: an ensemble of decision trees, typically more robust and less prone to overfitting than a single tree.

### Pipeline construction validation

Each pipeline was validated to confirm it was built correctly before training:

- Every model defined in `model` has a corresponding pipeline in `baseline_pipeline`, with matching keys.
- Each pipeline contains exactly two named steps: `preprocessing` and `model`.
- The `model` step in each pipeline references the exact same estimator instance defined in the original `model` dictionary.

These checks confirm that no pipeline was built with the wrong estimator or missing the preprocessing stage before proceeding to training.### Baseline model pipelines

A separate machine-learning pipeline was created for each baseline model. Every pipeline contains two stages:

1. The fitted preprocessing configuration transforms the original customer features.
2. The selected classifier learns from the resulting feature matrix.

Using a consistent pipeline structure ensures that every baseline model receives the same preprocessing treatment. These pipelines will be evaluated using the same validation strategy and metrics to enable a fair comparison.

In [55]:
assert set(baseline_pipeline.keys()) == set(model.keys())

for model_name, pipeline in baseline_pipeline.items():
    assert "preprocessing" in pipeline.named_steps
    assert "model" in pipeline.named_steps

    assert pipeline.named_steps["model"] is model[model_name]

print("All baseline pipelines were created correctly.")

All baseline pipelines were created correctly.


## Fit baseline models

Each baseline pipeline is fitted using only the untransformed training data. During fitting, the pipeline first learns the preprocessing parameters from `X_train` and then trains its classifier using the transformed features.

The test set is not used during this stage because it must remain untouched for the final evaluation. Fitting all baseline models through the same pipeline structure ensures consistent preprocessing and a fair comparison.